# PSTAT100 Data Science Concepts and Analysis
## Assignment 2

**Author:** 
**Date:** 

---

#### Submission Instructions

This assignment will be **due for submission on Sunday May 10th at 11:59PM**.  Please ensure you submit a `.pdf` file to Canvas by the posted time, no other file types will be accepted.  Please see the course website for some information regarding exporting Jupyter notebooks to pdf, and if you are having exporting problems discuss them with a member of the teaching staff in their office hours.

___

#### Collaboration

You are encouraged to work on your assignments independently.  Please only collaborate with others should you require assistance with more challenging problems. Should you need to collaborate with others, please note their names here:

> **Collaborators:**
___

#### Agent Usage

Additionally, you are permitted to use resources such as ChatGPT and Claude to help you with your homework assignments and to enhance your learning experience. Please make a note of any agents you have used in this submission here:

> **Agents:** 

**WARNING:** Please ensure you check and understand the outputs these agents produce.  Should your solution show evidence of abuse of AI usage (i.e. use external data not detailed in this assignment, use methods not covered in class) your solution will receive a score of 0.

---

## 1. Introduction

In this assignment we will be exploring achievement gaps in California school districts using a data set made available by [The Educational Opportunity Project at Stanford University](https://edopportunity.org).  This data was last updated in February 2026.  They have provided the article detailing their nationwide analysis as well as codebooks and data information on their [project page](https://edopportunity.org/opportunity/data/downloads/#documentation-6).

Throughout this assignment we will:

1. Review data documentation to:
   1. Identify population, sampling frame, sample.
   2. Assess scope of inference.
2. Prepare data by:
   1. Importing data.
   2. Manipulating `pandas` data frames.
   3. Merging data frames.
3. Performing exploratory data analysis by:
   1. Producing scatter plots.
   2. Multiplots.
   3. Univariate and multivariate analysis.
   4. Aggregation and tabulation.
   5. Visualizing trends.
4. Practice good communication by:
   1. Commenting on tables and figures.
   2. Summarizing results.
   3. Proposing hypotheses.



---

## 2. Background

Gender achievement gaps in education has been a topic of much interest over the years.  A controversial [article](https://www.jstor.org/stable/1684489) published in Science in 1980 argued that this pattern was due to an 'innate' difference in ability.  This article introduced bias by focussing primarily on mathematics (in which boys historically performed better) rather than reading and language (in which girls historically performed better).  Such views persisted in part because studying systematic patterns in achievement nationwide was a challenge due to differential testing standards across school districts and the general lack of availability of large-scale data.

The growth of data-driven research revealed socioeconomic drivers of achievement gaps. The [Educational Opportunity Project at Stanford University](https://edopportunity.org/) is a publicly available database on academic achievement and educational opportunity in U.S. schools. The database is part of a broader initiave aiming to improve educational opportunity by enabling researchers and policymakers to identify systemic drivers of disparity.

> The database includes a range of detailed data on educational conditions, contexts, and outcomes in school districts and counties across the United States. It includes measures of academic achievement and achievement gaps for school districts and counties, as well as district-level measures of racial and socioeconomic composition, racial and socioeconomic segregation patterns, and other features of the schooling system.

The database standardizes average test scores for schools 10,000 U.S. school districts relative to national standards to allow comparability between school districts and across grade levels and years. The test score data come from the U.S. Department of Education. In addition, multiple data sources (American Community Survey and Common Core of Data) are integrated to provide district-level socioeconomic and demographic information.

The following [study published in 2018](https://cepa.stanford.edu/content/gender-achievement-gaps-us-school-districts) identified from this data the following persistent patterns across grade levels 3 - 8 and school ears from 2008 through 2019:

* a consistent reading and language achievement gap favoring girls;
* *no* national math achievement gap on average; and
* local math achievement gaps that depend on the socioeconomic conditions of school districts.

In this assignment we shall work with selected portions of the database.  In this assignment we shall be making use of the following python libraries:

In [1]:
# Libraries
import numpy as np
import pandas as pd
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns

# Figure formatting
sns.set_style("whitegrid")
sns.set_palette("Set2")

---

## 3. Data Import and Understanding

We begin by downloading the required data files which can be found on Canvas:

1. `seda_geodist_long_gcs_6.0,csv` - contains test score data.
2. `seda_cov_geodist_long_6.0.csv` - contains the socioeconimic and demographic covariate data.

The data was sourced from The Educational Opportunity Project at Stanford University [website](https://edopportunity.org).  On this site scroll down to *Our Projects* and under *The 2009-2019 Educational Opportunity Project 6.0* click **Get The Data**.  After confirming their privacy agreement we are greeted by a large collection of codebooks and `.csv` files, some for the test data and some for the covariate data.  We note the following details about our chosen data:

1. `geodist` - Geographic District — the unit of observation is a geographic school district (a geographic catchment area, not just the administrative LEA)
2. `long` - Long form — data is structured with one row per district × year × grade × subject combination
3. `gcs` - Grade Cohort Standardized scale -  standardized with subject and across grades, relative to the average of four cohorts who were in 4th grade in 2009, 2011, 2013, and 2015. It's interpretable as an effect size — a district mean of 0.5 means the average student scored about half a standard deviation above the national reference cohort in that same grade.

Use the chunk below to load the test and covariate data as `test_raw` and `cov_raw`:

In [2]:
# Load raw data
test_raw = pd.read_csv(
    "data/seda_geodist_long_cs_6.0.csv"
)
cov_raw = pd.read_csv(
    "data/seda_cov_geodist_long_6.0.csv"
)

### 3.1 Test Score Data

Lets start by inspecting the test score data.  We start by looking at the `shape` of the data and printing the first few rows:

In [3]:
# Print test data shape
print("Test data shape:", test_raw.shape)
# View first few rows  
test_raw.head()

Test data shape: (1229235, 88)


,sedalea,sedaleaname,subject,grade,year,fips,stateabb,multi_comp,cs_mn_all,cs_mn_se_all,...,cs_mn_wht,cs_mn_se_wht,cs_mn_se_adj_wht,tot_asmt_wht,flag_estasmt_wht,cs_mn_wng,cs_mn_se_wng,cs_mn_se_adj_wng,tot_asmt_wng,flag_estasmt_wng
0,100005,Albertville City,mth,3,2009,1,AL,0,-0.437369,0.064553,...,-0.279577,0.096796,0.108811,177.0,0.0,NaN,NaN,NaN,NaN,NaN
1,100005,Albertville City,rla,3,2009,1,AL,0,-0.059948,0.064630,...,0.126219,0.100102,0.108965,178.0,0.0,NaN,NaN,NaN,NaN,NaN
2,100005,Albertville City,mth,3,2010,1,AL,0,0.040738,0.091833,...,0.207992,0.138220,0.142973,192.0,0.0,NaN,NaN,NaN,NaN,NaN
3,100005,Albertville City,rla,3,2010,1,AL,0,-0.034226,0.055393,...,0.129998,0.076952,0.082784,193.0,0.0,NaN,NaN,NaN,NaN,NaN
4,100005,Albertville City,mth,3,2011,1,AL,0,-0.000406,0.091107,...,0.225900,0.163970,0.171640,185.0,0.0,NaN,NaN,NaN,NaN,NaN


The data has ~1.2M observations and 88 columns.  We will only be considering a selection of these columns but we should spend some time understanding the labelling convention.  Reading the [documentation](https://stacks.stanford.edu/file/xh833nn4025/SEDA_documentation_6.0.pdf) we see that the test score means are named `cs_mn_...` with an abbreviation indicating subgroup (such as mean score for all `cs_mn_all`, for males `cs_mn_mal`, and so on).   

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q1.1 Interpreting test score values

Please interpret the average math test score for all 3rd grade studentsin Albertville City School District in 2009 (located in the first row of the data set).

---

**Solution:**

### 3.2 Covariate Data

Next we inspect the covariate data in the same way:

In [4]:
# Print covariate data shape
print("Covariate data shape:", cov_raw.shape)
# View first few rows
cov_raw.head()

Covariate data shape: (829097, 99)


,sedalea,sedaleaname,year,grade,fips,stateabb,gslo,gshi,urban,suburb,...,single_momasn,single_momblk,single_momhsp,single_momnam,single_momwht,seswhtblk,seswhthsp,seswhtasn,seswhtnam,urbanicity
0,100005,Albertville City,2009,3.0,1,AL,Pre-Kindergarten,12.0,0.0,0.0,...,NaN,NaN,0.199878,NaN,0.166134,NaN,1.069453,NaN,NaN,Town
1,100005,Albertville City,2009,4.0,1,AL,Pre-Kindergarten,12.0,0.0,0.0,...,NaN,NaN,0.199878,NaN,0.166134,NaN,1.069453,NaN,NaN,Town
2,100005,Albertville City,2009,5.0,1,AL,Pre-Kindergarten,12.0,0.0,0.0,...,NaN,NaN,0.199878,NaN,0.166134,NaN,1.069453,NaN,NaN,Town
3,100005,Albertville City,2009,6.0,1,AL,Pre-Kindergarten,12.0,0.0,0.0,...,NaN,NaN,0.199878,NaN,0.166134,NaN,1.069453,NaN,NaN,Town
4,100005,Albertville City,2009,7.0,1,AL,Pre-Kindergarten,12.0,0.0,0.0,...,NaN,NaN,0.199878,NaN,0.166134,NaN,1.069453,NaN,NaN,Town


In the SEDA data scores are *aggregated* to the district level by grade for multiple years. 

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q1.2 Observational Units

We are interested in determining and comparing the *observational units* of the two data frames.  A good rule of thumb for determining the observational units of a data frame is to consider: **what is the minimum number of columns whose combination is needed to uniquely identify rows**?  For example, in our `test_raw` data the combination of required rows is:

```
sedalea + subject + grade + year
```

Identify the combination of columns needed for `cov_raw` and comment on whether this is different to `test_raw`.  If our aim is to combine these  two data frames, discuss what the consequences might be for future merges.

---
**Solution:**

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q1.3 Sample Characteristics

Answer the following questions about the sampling design.  Note, you should not need to dig through the documentation to answer these questions.

1. What is the relevant population for the data sets?
2. Discuss the likely proportion of the population captured by the data (*Hint:* See the background section on this [page](https://edopportunity.org/opportunity/methods/)).
3. Comment on what kind of data set you suspect this is (administrative, typical sample, census).
4. In light of our description of the sample characteristics, what is the scope of inference for this data set.

---
**Solution:**

---

## 4. Data Preparation

Our goal will be to examine the relationship between gender achievement gaps and socioeconomic measures for school districts in California in 2019.  In order to do this, the following manipulations of the imported data are required:

1. Isolation data for CA in 2019.
2. Selecting columns of interest.
3. Filtering out non-urban districts.
4. Merging the covariate data with the test data.
5. Tidying the resulting data.

You will work with the following variables from each data set:

- **Test Score Data:**
  - District ID - `sedalea`
  - District name - `sedalename`
  - Grade - `grade`
  - Year - `year`
  - Test subject - `subject`
  - Estimated male-female gap - `cs_mn_mfg`
- **Covariate Data:**
  - District ID - `sedalea`
  - Locale:
    - Urban: `urban`
    - Suburban: `suburban`
    - Town: `town`
    - Rural: `rural`
  - Grade - `grade`
  - Socioeconomic statuss (all demographic groups) - `sesall`
  - Log median income (all demographic groups) - `lninc50all`
  - Poverty rate (all demographic groups) - `povertyall`
  - Unemployment rate (all demographic groups) - `unempall`
  - SNAP benefit rate (all demographic groups) - `snapall`



<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q2.1 Isolate CA and Year

Filter the data to create two new data sets, called `test_raw_ca` and `cov_raw_ca` respectively which contain data for California in 2019.  Make sure you use the `copy` function when creating these data sets so that they are not linked to your original data files.

**Hint:** Use `columns` to inspect the columns of the data frames.  The state is specified in `stateabb` and the year in `year`.

---
**Solution:**

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q2.2 Isolate variables of interest

We have included the column labels for all of the variables of interest listed above.  First create two lists `test_cols` and `cov_cols` of all of these labels.  Use these lists to isolate the columns of interest in both data frames.  

---
**Solution:**

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q2.3 Merge data frames

Use `merge` to combine the data frames.  Please read the [documentation](https://pandas.pydata.org/docs/reference/api/pandas.merge.html) top understand this function if you are unfamiliar with its use.  

**Hint:** Consider Q1.2 where we discussed the columns required to identify observations.  Which of these are in common across both data frames?

---
**Solution:**

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q2.4 Locale Data

We would like to combine the various locale columns into a single column.  The raw data is formatted such that each observation has proportions in (potentially multiple) locale columns depending on how it is best characterized.  For example:

1. Remove all observations with no data in all four locale columns.
2. Define a new column `locale` that gives which local column has the highest value by using the `pandas` function `idxmax(axis=1)`.
3. Drop the original locale columns.

---
**Solution:**

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q2.5 Rename columns

Rename the columns using the following list of column names:

In [5]:
# New column names
new_col_names = [
    "District ID", "District", "Year", "Grade", "Subject", "Gender Gap",
    "Socioeconomic Index", "log(Median Income)", "Poverty Rate",
    "Unemployment Rate", "SNAP Rate", "Locale"
        ]

Define our final data frame `data` by making a copy of `raw_data` and inspect the first few rows.

---
**Solution:**

## 5. Missingness

Gap estimates were not calculated for certain grades in certain districts due to small smample sizes (i.e. not enough individual tests were recorded).

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q3.1 Missingness Analysis

1. Create a missingness matrix plot using `matrix` from `missingno`.
2. Create a missingness table showing the count and percentage of missing values for each variable grouped by subject.
3. Determine what proportion of districts have missing gap estimates for one or both test subjects for at least one grade level.
4. Comment on whether you expect that this missingness is related to any particular district attributes.
---
**Solution:**

---

## 6. Exploration

### 6.1. Gender Gap and Socioeconomic Factors

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q6.1 Relationships between gender gap and socioeconomic factors

1. Using `seaborn` produce a panel of scatterplots showing the relationship between estimated gender gap and socioeconomic factors (`Socioeconomic Index`, `log(Median Income)`, `Poverty Rate`, `Unemployment Rate` and `SNAP Rate`) for all grade levels with points colored by test subject (5 plots in a row).

2. Provide your interpretation of the plot.  Comment on the gap for reading and math and how it seems to change depending on economic factors.  For example, does better economic factors increase the gaps?

---
**Solution:**

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q6.2 Splitting by Grade

Does the patter shown in the plot above persist within each grade level? Modify your previous code to show these relationships by grade level by generating a panel of scatter plot (5 columns for each factor, 5 rows for each grade).  Make sure that you adjust your formatting so that your results are clear and legible.

---
**Solution:**

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q6.3 Gaps shifting across grade level

Constuct a 2x5 panel of scatter plots showing estimated achievement gap against each of the 5 socioeconomic variables, with one row per test subject.  Display grade using a color gradient.  Do the gaps seem to shift with grade level?

---

**Solution:**


Yes, the scatter shifts from dark to light as the estimated gap decreases for both subjects and all socioeconomic variables.  This indicates that as grade level increases, the gap increasingly favors girls in both math and reading language.

---

While the magnitude of the achievement gaps seem to depend very slightly on grade level, the relationship between achievement gap and socioeconomic factors does not differ from grade to grade.  We will now focus on the average relationship between estimated achievement gap and median income after aggregativ across grade.  

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q6.4 Aggregating by Grade

Compute a new data frame `data_agg` by grouping by `District ID`, `District`, `Locale` and `Sbject` and computing the mean for the remaining variables.  We drop `Grade` since we no longer need it.

---
**Solution:**

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q6.5 District Average Gaps

Construct a scatterplot of the average estimated gap against log(Median Income) by subject for each district and add trend lines.

---

**Solution:**

---

Let's try to capture this pattern in tabular form.  The call below adds an `Income Bracket` variable by cutting the median income into 8 contiguous intervals using `cut`, and tabulates the average socioeconomic measures and estimated gaps across districts by income bracket.  Note that with respect to the gaps, this displays the pattern that is shown visually in the figure above.

> **Note:** Make sure that you remove the """ as this is making this code a comment and therefore it will not execute.  This is to avoid any potential rendering problems!

In [6]:
"""
data_agg['Income Bracket'] = pd.cut(np.e**data_agg['log(Median Income)'], 8)
data_agg.head()
"""

"\ndata_agg['Income Bracket'] = pd.cut(np.e**data_agg['log(Median Income)'], 8)\ndata_agg.head()\n"

In [7]:
"""
data_agg.pivot_table(
    values='Gender Gap',
    index='Subject',
    columns='Income Bracket',
    aggfunc='mean'
).round(4)
"""

"\ndata_agg.pivot_table(\n    values='Gender Gap',\n    index='Subject',\n    columns='Income Bracket',\n    aggfunc='mean'\n).round(4)\n"


<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q6.6 Proportion of districts with math gap

What proportion of districts in each income bracket have an average estimated math achievement gap favoring boys.  Answer this question by performing the following steps:

1. Append an indicator variable `Math Gap Favoring Boys` to `data_agg` that records whether the average estimated math gap favors boys more than 0.1 standard deviations relative to the national average.
2. Compute the proportion of districts in each income bracket for which the indicator is true: group by bracket and take the mean.

---
**Solution:**

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q6.7 Statewide Averages

Calculate a few statewide averages to get a sense of how some of the patterns above4 compare with the state as a whole:

1. Compute the statewide average estimated achievement gaps.
2. Compute the proportion of districts in the state with a math gap favoring boys.
3. Compute the proportion of districts in the state with a math gap favoring girls (this needs a new indicator).

---
**Solution:**

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q6.8 Socioeconomic Factor Correlation

To inspect for relationships between the covariates produce a `pairplot` showing the interactions of the 5 socioeconomic variables.  Comment on your results.  Is this what you expected?

---
**Solution:**

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q6.9 Summarizing

1. Write a brief summary (3-5 sentences) of your exploratory analysis.  What have you discovered about educational achievement gaps in California school districts.  
2. Discuss if you have identified a correlation between socioeconomic factors and achievement gaps in your 

---

**Solution:**

---

## Submission

1. Save file to confirm all changes are on disk
2. Run *Kernel > Restart & Run All* to execute all code from top to bottom
3. Save file again to write any new output to disk
4. Export your notebook as a pdf (either through latex or as html before saving as a pdf).
5. Submit your pdf to Canvas.